# 读取 HETU 量化研究平台结果

与网页读取同一已发布产物：只调用 `ExperimentStore.get_result` 和 `export_result`，
直接读取结果目录内的结构化表，不复制公式、不重新计算。
运行时选择量化环境（quant/.venv）即可；根目录默认仓库下 `.hetu/quant`，
测试或迁移时可用环境变量 `HETU_QUANT_ROOT` 指向其他根目录。

In [ ]:
import os
from pathlib import Path

root = Path(os.environ.get("HETU_QUANT_ROOT", ".hetu/quant"))
print(root)

In [ ]:
from hetu_quant.experiments import ExperimentStore

store = ExperimentStore(root)
result_ids = store.list_results()
assert result_ids, "根目录下没有已发布结果"
result_id = result_ids[-1]
print(result_id)

In [ ]:
import pandas as pd

result = store.get_result(result_id)
result_dir = root / "results" / result.result_id
tables = {
    path.stem: pd.read_parquet(path)
    for path in sorted(result_dir.glob("*.parquet"))
}
slots = tables["slots"]
boards = tables["boards"]
cards = tables["cards"]
print("status:", result.status)
print("seats:", len(slots), "distinct:", slots["code"].nunique())

In [ ]:
boards[(boards["board_kind"] == "industry") & boards["selected"]][
    ["board_id", "name", "rank", "score"]
]

In [ ]:
from hetu_quant.reporting import export_result

zip_path = export_result(result, root / "exports")
print(zip_path)